## STUDY A: IMPORT LIBRARIES, LOAD, AND FORMAT DATA

In [37]:

# import key libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)
# ("new")

# full dataset

# set working directory & load FULL data from local storage
setwd("C:/Users/jacob/water_current_MA/data/dual_generalization")
df_A_full <- read.csv("PCA_error_projectile_experiment _S1-projectile_experiment_S2-projectile_experiment_S1_3.0M-projectile_experiment 1_S2_3.0M-projectile_experiment 1_speed_neg3_0_neg2_0_0_0.csv")


# Format data types 
df_A_full$ppid_full <- factor(df_A_full$ppid_full)
df_A_full$speed_label <- factor(df_A_full$speed_label)
df_A_full$target_x_label <- factor(df_A_full$target_x_label)
df_A_full$phase <- factor(df_A_full$phase)

df_A_full$set_order <- factor(df_A_full$set_order)
# change set_order names to reflect thesis, group 1 and group 2 labels
levels(df_A_full$set_order) <- c("group_1","group_2")

# ensure consistent levels
df_A_full$target_x_label <- factor(df_A_full$target_x_label, 
                                 levels = c("L60", "L30", "R30", "R60"))


# load the smaller, early late (first and last 2 trials/phase) dataset
df_A_early_late_2 <- read.csv("early_late_2_phase_PCA_error_projectile_experiment _S1-projectile_experiment_etc_speed_neg3_0_neg2_0.csv")

# Format data types 
df_A_early_late_2$ppid_full <- factor(df_A_early_late_2$ppid_full)
df_A_early_late_2$speed_label <- factor(df_A_early_late_2$speed_label)
df_A_early_late_2$target_x_label <- factor(df_A_early_late_2$target_x_label)
df_A_early_late_2$phase <- factor(df_A_early_late_2$phase)

df_A_early_late_2$set_order <- factor(df_A_early_late_2$set_order)
# change set_order names to reflect thesis, group 1 and group 2 labels
levels(df_A_early_late_2$set_order) <- c("group_1","group_2")

# ensure consistent levels
df_A_early_late_2$target_x_label <- factor(df_A_early_late_2$target_x_label, 
                                     levels = c("L60", "L30", "R30", "R60"))

# STUDY A: Training Phase 1: early and late 2 trials

Here we compared the mean minimum metric error for the first and last two trials per target during the naive, initial Training Phase 1. We conducted within-subjects ANOVAs on minimum metric error with trial set and target as within-subject factors. Each ANOVA was conducted across two target pair groups (i.e., Group 1 and Group 2) and separately for two water current experiments. Consequently, we ran a total of 4 within-subjects ANOVAs (two groups, two experiments). Holm's correction was reported for the two group ANOVAs in each separate experiment. The purpose of this analysis was to examine robust learning following practice compensating for the water current perturbation.


Speed 3.0 m/s

The interaction was not significant across either group, suggesting that both targets within each pair produced comparable learning magnitudes from early to late training. However, the main effect of trial set was highly significant for both Group 1 and Group 2 after applying Holm's correction, producing large generalized eta-squared values of 0.245 and 0.304, respectively. This demonstrates a robust error reduction following practice compensating for the water current that was consistent across both target pair groups.


Speed 2.0 m/s

Crucially, the interaction in this experiment was significant in Group 1, but both targets showed significant error reduction--R60 drove larger errors as expected. Nonetheless, here we also observed significant main effects of trial set for both groups after applying Holm's correction. Effect sizes were similar to the previous experiment, with Group 1 yielding a large generalized eta-squared value of 0.355, while Group 2 showed a generalized eta-squared value of 0.311. Again, we observed a significant learning effect following practice in compensating for the water current.

ANOVA results are below.

In [39]:

# Isolate training phase
df_A_early_late_2_t1 <- df_A_early_late_2[df_A_early_late_2$phase == 'training_1',]
speeds_list <- unique(df_A_early_late_2_t1$speed_label)
set_order_list <- unique(df_A_early_late_2_t1$set_order)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_A_early_late_2_t1[df_A_early_late_2_t1$speed_label == speed & df_A_early_late_2_t1$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = c("target_x_label", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.2255671 
Holm-Adjusted trial_set Main Effect p-value:  1.964611e-06 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F  ges p.value
1           target_x_label 1, 19 333.79 49.81 *** .260   <.001
2                trial_set 1, 19 411.89 50.10 *** .304   <.001
3 target_x_label:trial_set 1, 19 288.23      1.57 .009    .226
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     32.1 4.54 19   7.078  <.0001

Results are averaged over the levels of: target_x_label 

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.1702784 
Holm-Adjusted trial_set Main Effect p-value:  4.587144e-06 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE        

# STUDY A: Training Phase 2: early and late 2 trials

Similar to the analysis above, we conducted a within-subjects ANOVA on mean minimum metric error for the first and last two trials per target, but now during the second training phase. Again, we conducted four within-subjects ANOVAs on minimum metric error with trial set and target as within-subject factors, separately for both water current experiments. Holm's correction was also reported for the two group ANOVAs in each separate experiment. The purpose of this analysis was to examine any ongoing learning after completing Training Phase 1.


Speed 3.0 m/s

The interaction was not significant across either group, suggesting that both novel targets within each pair produced comparable learning magnitudes across this phase. Importantly, because the main effect of trial set was not significant, this demonstrates that learning had saturated by the onset of this phase. This hints at a transfer effect, where learning in the first training phase results in error saturation for novel targets in the second phase. Overall, because we did not observe a main effect or interaction involving trial set during Training Phase 2 for this experiment, this illustrates learning saturation, and thus no ongoing learning was detected.


Speed 2.0 m/s

Interestingly, the interaction in this experiment was significant for both groups. Post-hoc analyses revealed that this effect was driven by the more challenging, upstream-rightward targets. However, only the R60 target in Group 2 remained significant following the post-hocs. This demonstrates a continuation of learning for arguably the most challenging target in this task for this group. Because participants who experienced this target during the second training phase must have switched from R30 in Training Phase 1. This apparent ongoing learning could be driven by transitioning from an easier to a more challenging target, as the other group only showed a small trend with the R30 target after post-hocs. Nonetheless, most targets did not show further learning.

ANOVA results are below.

In [41]:

# Isolate training phase
df_A_early_late_2_t2 <- df_A_early_late_2[df_A_early_late_2$phase == 'training_2',]
speeds_list <- unique(df_A_early_late_2_t2$speed_label)
set_order_list <- unique(df_A_early_late_2_t2$set_order)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_A_early_late_2_t2[df_A_early_late_2_t2$speed_label == speed & df_A_early_late_2_t2$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = c("target_x_label", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.3581706 
Holm-Adjusted trial_set Main Effect p-value:  0.5017309 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F  ges p.value
1           target_x_label 1, 19 375.12 34.55 *** .235   <.001
2                trial_set 1, 19 544.53      1.40 .018    .251
3 target_x_label:trial_set 1, 19 242.07      1.95 .011    .179
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Neither effect survived Holm correction. Skipping post-hocs.

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.3581706 
Holm-Adjusted trial_set Main Effect p-value:  0.9049117 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F   ges p.value
1           target_x_label 1, 24 396.01 26.54 ***  .214   <.001
2                trial_set 1, 24 188.48      0.01 <.001    .905

# STUDY A: Transfer Test

Here, we were interested in comparing initial errors for each target during Training Phase 1 versus Training Phase 2. As a reminder, participants were exposed to a counterbalanced target pair during each phase. This design allows us to compare initial target errors when participants are naive to the perturbation (i.e., early trials of Training Phase 1) to those when they have already experienced the impact of the perturbation (i.e., early Training Phase 2), albeit in a spatially distinct region of the workspace. The purpose of this analysis was to measure generalization and transfer. Transfer would be observed if initial target errors during Training Phase 2 are significantly lower than initial errors during Training Phase 1. This would suggest participants have developed a generalizable internal model of the ball-water interactions, allowing them to generate motor commands closer to task success than their naive counterparts. To address this hypothesis, we decided to conduct a series of four one-tailed Welch's t-tests for both experiments, since our main question involved a between-subjects effect of phase across each target. We chose a one-tailed test because we have theoretical justification to expect a directional learning effect. This approach not only aligns with our expectations, but also provides enhanced power for detecting transfer. Again, Holm's correction was applied due to multiple comparisons.

Because experienced participants outperformed naive participants in all four t-tests for both water current experiments, this undeniably demonstrates strong transfer. Importantly, near-complete transfer (i.e., errors near 0) was observed for most of the downstream targets in both experiments, while strong but incomplete transfer was observed for the upstream targets. Nonetheless, the consistent and strong transfer effect observed broadly across the workspace provides evidence for a generalizable internal representation of ball-water dynamics, facilitating immediate error reduction to novel target locations.

One-tailed Welch's t-test results are below. 

In [43]:

# Filter for Transfer (Early trials only)
df_A_transfer <- df_A_early_late_2[df_A_early_late_2$trial_set == 'early',]
df_A_transfer$phase <- factor(df_A_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_A_transfer$speed_label)
target_list <- unique(df_A_transfer$target_x_label)

df_agg <- aggregate(
  flip_min_distance_xPCA_mean_bc ~ ppid_full + phase + target_x_label + speed_label,
  data = df_A_transfer,
  FUN = mean
)


for (s in speeds_list) {

    p_vals <- c()
    
    for (t in target_list) {
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_agg[df_agg$speed_label == s & df_agg$target_x_label == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset, alternative = "greater")
          print(res)

          # show sd
          print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$phase, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'experiment:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'experiment:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}



--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 4.5573, df = 27.916, p-value = 4.676e-05
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 21.39598      Inf
sample estimates:
mean in group training_1 mean in group training_2 
               31.048690                -3.093021 

training_1 training_2 
  30.09605   16.46011 

Cohen's d

d estimate: 1.453899 (large)
95 percent confidence interval:
    lower     upper 
0.7745197 2.1332790 


--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L30 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 2.1333, df = 39.246, p-value = 0.01961
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 3.796919      Inf
sample estima

## STUDY B: IMPORT LIBRARIES, LOAD, AND FORMAT DATA

In [49]:
# import key libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)

# set working directory & load FULL data from local storage
setwd("C:/Users/jacob/water_current_MA/data")

# load the early late (first and last 2 trials/phase) dataset for dual target training
df_B_early_late_2 <- read.csv("spatial_generalization/early_late_phase_2_PCA_error_novel_TU_3.0M_spatial_generalization_novel_TD_3.0M_spatial_generalization_speed_neg3_0.csv")

# load full dfs
df_B_full <- read.csv("spatial_generalization/PCA_error_novel_TU_3.0M_spatial_generalization_novel_TD_3.0M_spatial_generalization_speed_neg3_0_0_0.csv")

df_BA_trials_75_vs_225 <- read.csv("spatial_generalization/df_trials_75_vs_225.csv")

df_B_names <- c("df_B_early_late_2", "df_B_full", "df_BA_trials_75_vs_225")
df_B_list <- mget(df_B_names)

# loop through all df to adjust data types
for (i in seq_along(df_B_list)) {

    df <- df_B_list[[i]]

    # Format data types 
    df$ppid_full <- factor(df$ppid_full)
    # df$speed_label <- factor(df$speed_label)
    df$target_x_label <- factor(df$target_x_label)
    df$phase <- factor(df$phase)
    
    df$set_order <- factor(df$set_order)
    # change set_order names to reflect thesis, group 1 and group 2 labels
    levels(df$set_order) <- c("group_1","group_2")
    
    # ensure consistent levels
    df$target_x_label <- factor(df$target_x_label, 
                                         levels = c("L60", "L30", "R30", "R60"))

    df_B_list[[i]] <- df
    }

# list2env(df_list, envir = .GlobalEnv)




# STUDY B: Training Phase 1: early and late 2 trials

In [51]:
# Isolate training phase
df_B_early_late_2_t1 <- df_B_early_late_2[df_B_early_late_2$phase == 'training_1',]
# speeds_list <- unique(df_early_late_2_single_t1$speed_label)
target_list <- unique(df_B_early_late_2_t1$target_x_label)

df_agg <- aggregate(
  flip_min_distance_xPCA_mean_bc ~ ppid_full + trial_set + target_x_label,
  data = df_B_early_late_2_t1,
  FUN = mean
)

p_vals <- c()

for (t in target_list) {

    
    # for (t in target_list) {
    #     print(t)
  
        cat("\n--- RUNNING ANALYSIS FOR TARGET:", t, "---\n")
          
        df_subset <- df_agg[df_agg$target_x_label == t,]
        df_subset <- droplevels(df_subset) 

        early = df_subset$flip_min_distance_xPCA_mean_bc[df_subset$trial_set == "early"]
        late = df_subset$flip_min_distance_xPCA_mean_bc[df_subset$trial_set == "late"]
          
        res <- t.test(x = early,
                      y = late,
                      data = df_subset, 
                      paired=TRUE,
                      alternative="greater")
        print(res)

          # show sd
        print('sd')
        print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$trial_set, sd, na.rm = TRUE))

        print('mean')
        print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$trial_set, mean, na.rm = TRUE))
        
        
          # effect sizes
        print(cohen.d(early, late, paired=TRUE))
        
          # Grab p-value & store
        p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat('Raw p-vals:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat('Adjusted p-vals:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)



--- RUNNING ANALYSIS FOR TARGET: R60 ---

	Paired t-test

data:  early and late
t = 10.757, df = 19, p-value = 8.022e-10
alternative hypothesis: true mean difference is greater than 0
95 percent confidence interval:
 48.20149      Inf
sample estimates:
mean difference 
       57.43371 

[1] "sd"
   early     late 
24.87654 24.13968 
[1] "mean"
   early     late 
76.51333 19.07962 

Cohen's d

d estimate: 2.342606 (large)
95 percent confidence interval:
   lower    upper 
1.489570 3.195642 


--- RUNNING ANALYSIS FOR TARGET: L60 ---

	Paired t-test

data:  early and late
t = 6.7376, df = 15, p-value = 3.332e-06
alternative hypothesis: true mean difference is greater than 0
95 percent confidence interval:
 24.07699      Inf
sample estimates:
mean difference 
       32.54477 

[1] "sd"
   early     late 
14.56314 13.08863 
[1] "mean"
   early     late 
44.75978 12.21501 

Cohen's d

d estimate: 2.35038 (large)
95 percent confidence interval:
    lower     upper 
0.9685218 3.7322391 

Raw

# STUDY B: Training Phase 2: early and late 2 trials

In [53]:
# Isolate training phase
df_B_early_late_2_t2 <- df_B_early_late_2[df_B_early_late_2$phase == 'training_2',]
# speeds_list <- unique(df_early_late_2_single_t1$speed_label)
target_list <- unique(df_B_early_late_2_t2$target_x_label)

df_agg <- aggregate(
  flip_min_distance_xPCA_mean_bc ~ ppid_full + trial_set + target_x_label,
  data = df_B_early_late_2_t2,
  FUN = mean
)

p_vals <- c()

for (t in target_list) {

    
    # for (t in target_list) {
    #     print(t)
  
        cat("\n--- RUNNING ANALYSIS FOR TARGET:", t, "---\n")
          
        df_subset <- df_agg[df_agg$target_x_label == t,]
        df_subset <- droplevels(df_subset) 

        early = df_subset$flip_min_distance_xPCA_mean_bc[df_subset$trial_set == "early"]
        late = df_subset$flip_min_distance_xPCA_mean_bc[df_subset$trial_set == "late"]
          
        res <- t.test(x = early,
                      y = late,
                      data = df_subset, 
                      paired=TRUE,
                      alternative="greater")
        print(res)

          # show sd
        print('sd')
        print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$trial_set, sd, na.rm = TRUE))

        print('mean')
        print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$trial_set, mean, na.rm = TRUE))
        
        
          # effect sizes
        print(cohen.d(early, late, paired=TRUE))
        
          # Grab p-value & store
        p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat('Adjusted p-vals:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)



--- RUNNING ANALYSIS FOR TARGET: L60 ---

	Paired t-test

data:  early and late
t = 1.1909, df = 19, p-value = 0.1242
alternative hypothesis: true mean difference is greater than 0
95 percent confidence interval:
 -3.476644       Inf
sample estimates:
mean difference 
       7.693372 

[1] "sd"
   early     late 
18.77165 16.43947 
[1] "mean"
   early     late 
9.123276 1.429904 

Cohen's d

d estimate: 0.4365177 (small)
95 percent confidence interval:
    lower     upper 
-0.340028  1.213063 


--- RUNNING ANALYSIS FOR TARGET: R60 ---

	Paired t-test

data:  early and late
t = 6.4963, df = 15, p-value = 5.047e-06
alternative hypothesis: true mean difference is greater than 0
95 percent confidence interval:
 26.96266      Inf
sample estimates:
mean difference 
        36.9277 

[1] "sd"
   early     late 
19.42574 20.24680 
[1] "mean"
   early     late 
60.89298 23.96528 

Cohen's d

d estimate: 1.86081 (large)
95 percent confidence interval:
    lower     upper 
0.8940174 2.8276027 


# STUDY B: Transfer

In [55]:

# Filter for Transfer (Early trials only)
df_B_transfer <- df_B_early_late_2[df_B_early_late_2$trial_set == 'early',]
df_B_transfer$phase <- factor(df_B_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_B_transfer$speed_label)
target_list <- unique(df_B_transfer$target_x_label)

df_agg <- aggregate(
  flip_min_distance_xPCA_mean_bc ~ ppid_full + phase + target_x_label,
  data = df_B_transfer,
  FUN = mean
)


p_vals <- c()
    
for (t in target_list) {
  
        cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR TARGET:", t, "---\n")
          
        df_subset <- df_agg[df_agg$target_x_label == t,]
        df_subset <- droplevels(df_subset) 
          
        res <- t.test(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset, alternative = "greater")
        print(res)

        # show sd
        print('sd')
        print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$phase, sd, na.rm = TRUE))
        
        # effect sizes
        print(cohen.d(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset))
        
    # Grab p-value & store
    p_vals <- c(p_vals, res$p.value)


    
    # show p-vals
    cat('Raw p-vals:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat('Adjusted p-vals:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}



--- RUNNING BETWEEN TRANSFER ANALYSIS FOR TARGET: R60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 2.1154, df = 33.99, p-value = 0.0209
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 3.134047      Inf
sample estimates:
mean in group training_1 mean in group training_2 
                76.51333                 60.89298 

[1] "sd"
training_1 training_2 
  24.87654   19.42574 

Cohen's d

d estimate: 0.6901226 (medium)
95 percent confidence interval:
      lower       upper 
-0.01126635  1.39151153 

Raw p-vals:
[1] 0.02090342
Adjusted p-vals:
[1] 0.02090342

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR TARGET: L60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 6.4135, df = 33.981, p-value = 1.26e-07
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent 

# STUDY A vs STUDY B

Matched Late Comparison

### Comparing performance across Study A and Study B on the same targets in Training Phase 1

In [57]:
# set dir and load dfs

setwd("C:/Users/jacob/water_current_MA/data")
df_A_early_late <- read.csv("dual_generalization/early_mid_late_S1_3.0M-projectile_experiment 1_S2_3.0M-projectile_experiment 1_speed_neg3_0.csv")
df_B_early_mid_late <- read.csv("spatial_generalization/early_mid_late_novel_TU_3.0M_spatial_generalization_novel_TD_3.0M_spatial_generalization_speed_neg3_0.csv")

# load libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)
library(tidyr)
library(forcats)

# label factors
factor_cols <- c('ppid_full','experiment','target_x_label','phase','trial_set')

# format columns in each df:

# STUDY A:
target_cols <- intersect(names(df_A_early_late), factor_cols)
df_A_early_late[target_cols] <- lapply(df_A_early_late[target_cols], as.factor)
# for factor comparison across these two df, we need to rename this df's trial_set range_-2_None to range_73_75
df_A_early_late$trial_set <- fct_recode(df_A_early_late$trial_set,
                                 "range_73_75" = "range_-2_None")
# add a new column to distinguish study A from study B
df_A_early_late <- df_A_early_late %>%
    mutate(study = factor("A"))


# STUDY B:
target_cols <- intersect(names(df_B_early_mid_late), factor_cols)
df_B_early_mid_late[target_cols] <- lapply(df_B_early_mid_late[target_cols], as.factor)
# drop trial_set "range_-2_None" as it is not in STUDY A dataset
df_B_early_mid <- df_B_early_mid_late %>%
  filter(trial_set != "range_-2_None") %>%
  mutate(trial_set = fct_drop(trial_set),
        study = factor("B")
        )
           
# now, we need to combine the dfs
combined_df <- bind_rows(df_A_early_late, df_B_early_mid)

# updated ppid_full column
combined_df <- combined_df %>%
  mutate(ppid_full = paste0(ppid_full, "_", study))

# conduct tests
df_agg <- aggregate(
  flip_min_distance_xPCA_mean_bc ~ ppid_full + trial_set + target_x_label + study,
  data = combined_df,
  FUN = mean
)

trial_set_list <- unique(df_agg$trial_set)
target_list <- unique(df_agg$target_x_label)

    p_vals <- c()


for (set in trial_set_list) {
    
    for (t in target_list) {

          # skip if inner targets
          if (t %in% c('L30', 'R30')) {
              next
              }
  
          cat("\n--- RUNNING ANALYSIS FOR TRIAL_SET x TARGET:", set, t, "---\n")
          
          df_subset <- df_agg[df_agg$trial_set == set & df_agg$target_x_label == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(flip_min_distance_xPCA_mean_bc ~ study, data = df_subset)
          print(res)

          # show sd
          print('sd')
          print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$study, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(flip_min_distance_xPCA_mean_bc ~ study, data = df_subset))

          # Grab p-value & store
          label <- paste(set, t, sep = "_")
          p_vals[label] <- res$p.value
    

    }

}

    # show p-vals
    cat('\n========================================\n')
    cat('Raw p-vals for ALL sets and targets:\n')
    print(p_vals)
    
    # adjusted p-vals (computed together as one family)
    cat('\nAdjusted p-vals:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
    cat('========================================\n')


--- RUNNING ANALYSIS FOR TRIAL_SET x TARGET: range_73_75 L60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by study
t = -0.92519, df = 33.998, p-value = 0.3614
alternative hypothesis: true difference in means between group A and group B is not equal to 0
95 percent confidence interval:
 -19.87401   7.43948
sample estimates:
mean in group A mean in group B 
       3.682259        9.899525 

[1] "sd"
      A       B 
22.5408 17.7783 

Cohen's d

d estimate: -0.3021606 (small)
95 percent confidence interval:
     lower      upper 
-0.9876270  0.3833057 


--- RUNNING ANALYSIS FOR TRIAL_SET x TARGET: range_73_75 R60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by study
t = 1.8789, df = 42.113, p-value = 0.06719
alternative hypothesis: true difference in means between group A and group B is not equal to 0
95 percent confidence interval:
 -1.127882 31.624600
sample estimates:
mean in group A mean in group B 
       36.71697        21.46861 

[1

# Study B: Comparing Early, Matched, and Late performance in Training Phase 1

In [59]:
df_B_early_mid_late <- read.csv("spatial_generalization/early_mid_late_novel_TU_3.0M_spatial_generalization_novel_TD_3.0M_spatial_generalization_speed_neg3_0.csv")

# load libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)
library(tidyr)
library(forcats)

factor_cols <- c('ppid_full','experiment','target_x_label','phase','trial_set')

# format columns in  df:
target_cols <- intersect(names(df_B_early_mid_late), factor_cols)
df_B_early_mid_late[target_cols] <- lapply(df_B_early_mid_late[target_cols], as.factor)

# initilize empty vectors
results_storage <- list()
raw_trial_set_p_vals <- c()
comparison_labels <- c()

target_list <- unique(df_B_early_mid_late$target_x_label)

# RUN MODELS & EXTRACT P-VALUES
for (t in target_list) {
  
  # skip if inner targets
  if (t %in% c('L30', 'R30')) {
    next
  }
    
  df_subset <- df_B_early_mid_late[df_B_early_mid_late$target_x_label == t,]
  df_subset <- droplevels(df_subset) 
  
  model_anova <- afex::aov_ez(
    id = "ppid_full",
    dv = "flip_min_distance_xPCA_mean_bc",
    data = df_subset,
    within = "trial_set",
    fun_aggregate = mean
  )
  
  label <- paste(t)
  results_storage[[label]] <- model_anova
  
  raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
  comparison_labels <- c(comparison_labels, label)
  
}
    
# Apply Holm corrections 
adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
names(adjusted_trial_set_p_vals) <- comparison_labels

# POST-HOCS AND PRINTING
for (label in comparison_labels) {
  
  model_anova <- results_storage[[label]]
  p_adj_main_trial_set <- adjusted_trial_set_p_vals[label]
  
  cat("\n========================================\n")
  cat("ANALYSIS FOR TARGET:", label, "\n")
  cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main_trial_set, "\n")
  cat("========================================\n")

  # Show descriptive stats
  df_subset <- df_B_early_mid_late[df_B_early_mid_late$target_x_label == label, ]
  cat("\nDescriptive stats:\n")
  print(aggregate(flip_min_distance_xPCA_mean_bc ~ trial_set, df_subset, function(x) c(mean = mean(x, na.rm=TRUE), sd = sd(x, na.rm=TRUE))))
  cat("\n")

  # Print Model
  print(model_anova)
  
  # Conditional logic based on Holm-corrected results
  if (p_adj_main_trial_set <= 0.05) {
    cat("\nSignificant Main Effect of trial_set. Running post-hocs:\n")
    print(pairs(emmeans::emmeans(model_anova, ~ trial_set), adjust="holm"))
  } else {
    cat("\nMain effect did not survive Holm correction. Skipping post-hocs.\n")
  }
}


ANALYSIS FOR TARGET: R60 
Holm-Adjusted trial_set Main Effect p-value:  1.697235e-13 

Descriptive stats:
      trial_set flip_min_distance_xPCA_mean_bc.mean
1 range_-2_None                            19.07962
2     range_0_2                            76.51333
3   range_73_75                            21.46861
  flip_min_distance_xPCA_mean_bc.sd
1                          27.54457
2                          27.08470
3                          28.57247

Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
     Effect          df    MSE         F  ges p.value
1 trial_set 1.95, 37.10 281.42 76.85 *** .542   <.001
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Sphericity correction method: GG 

Significant Main Effect of trial_set. Running post-hocs:
 contrast                    estimate   SE df t.ratio p.value
 range_.2_None - range_0_2     -57.43 5.34 19 -10.757  <.0001
 range_.2_None - range_73_75    -2.39 5.53 19  -0.432  0.6705
 range_0_2 - rang

# MALANOBIS-LIKE DISTANCE

In [61]:
# import key libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)

setwd("C:/Users/jacob/water_current_MA/data/mahalanobis_data")

# Load datasets
df_single <- read.csv("early_late_SINGLE_df.csv")
df_single$experiment <- factor("SINGLE")

df_dual <- read.csv("early_late_DUAL_df.csv")
df_dual$experiment <- factor("DUAL")

# Store data frames explicitly in a named list to avoid ls() collisions
data_list <- list(
  df_single = df_single,
  df_dual   = df_dual
)

# Function to format data types cleanly
format_datatypes <- function(df) {
    
    df$ppid_full <- factor(df$ppid_full)
    df$water_speed <- factor(df$water_speed)
    df$phase <- factor(df$phase)
    df$set_order <- factor(df$set_order)
    levels(df$set_order) <- c("group_1", "group_2")
    df$trial_set <- factor(df$trial_set)
      
    # Ensure consistent factor levels across all datasets
    df$target_id <- factor(df$target_id, levels = c("L60", "L30", "R30", "R60"))
      
    return(df)
}

# Apply formatting function to both data frames
data_list <- lapply(data_list, format_datatypes)

# Export formatted data frames back to the Global Environment
list2env(data_list, envir = .GlobalEnv)


<environment: R_GlobalEnv>

# STUDY A: Training Phase 1: early and late 2 trials

In [63]:
# Isolate dual target experiment training phase 1
df_dual_t1 <- df_dual[df_dual$phase == 'training_1',]
speeds_list <- unique(df_dual_t1$water_speed)
set_order_list <- unique(df_dual_t1$set_order)

# reset
p_adj_inter <- 1.0
p_adj_main <- 1.0


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_dual_t1[df_dual_t1$water_speed == speed & df_dual_t1$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "min_mahalanobis_distance",                  
            data = df_subset,                
            within = c("target_id", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_id:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_id), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.03569423 
Holm-Adjusted trial_set Main Effect p-value:  2.570485e-07 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
               Effect    df  MSE         F  ges p.value
1           target_id 1, 19 0.23 46.88 *** .242   <.001
2           trial_set 1, 19 0.27 66.32 *** .352   <.001
3 target_id:trial_set 1, 19 0.25    5.78 * .042    .027
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Significant Interaction. Running target-specific post-hocs:
target_id = L60:
 contrast     estimate    SE df t.ratio p.value
 early - late    0.681 0.126 19   5.414  <.0001

target_id = R30:
 contrast     estimate    SE df t.ratio p.value
 early - late    1.224 0.192 19   6.363  <.0001


ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.03569423 
Holm-Adjusted trial_set Main Effect p-value:  2.570485e-07 
Anova Table (Type 3 tests)

Response: min_mahalanobis_dista

# STUDY A: Training Phase 2: early and late 2 trials

In [65]:
# Isolate dual target experiment training phase 1
df_dual_t2 <- df_dual[df_dual$phase == 'training_2',]
speeds_list <- unique(df_dual_t2$water_speed)
set_order_list <- unique(df_dual_t2$set_order)

# reset
p_adj_inter <- 1.0
p_adj_main <- 1.0


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_dual_t2[df_dual_t2$water_speed == speed & df_dual_t2$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "min_mahalanobis_distance",                  
            data = df_subset,                
            within = c("target_id", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_id:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_id), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.6870235 
Holm-Adjusted trial_set Main Effect p-value:  0.01441427 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
               Effect    df  MSE       F  ges p.value
1           target_id 1, 19 0.36  7.75 * .118    .012
2           trial_set 1, 19 0.27 9.06 ** .104    .007
3 target_id:trial_set 1, 19 0.13    0.17 .001    .687
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate    SE df t.ratio p.value
 early - late    0.348 0.116 19   3.010  0.0072

Results are averaged over the levels of: target_id 

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.6138664 
Holm-Adjusted trial_set Main Effect p-value:  0.1490126 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
               Effect    df  MSE      F  ges p.value
1           target_id 1, 24 0.30 5.21 * .068    .0

# STUDY A: TRANSFER

In [67]:

# Filter for Transfer (Early trials only)
df_dual_transfer <- df_dual[df_dual$trial_set == 'early',]
df_dual_transfer$phase <- factor(df_dual_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_dual_transfer$water_speed)

# aggregate for means
df_dual_transfer_agg <- aggregate(
  min_mahalanobis_distance ~ ppid_full + phase + water_speed + target_id,
  data = df_dual_transfer,
  FUN = mean
)

for (s in speeds_list) {
    print(s)

    p_vals <- c()
    
    for (t in target_list) {
        print(t)
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_dual_transfer_agg[df_dual_transfer_agg$water_speed == s & df_dual_transfer_agg$target_id == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(min_mahalanobis_distance ~ phase, data = df_subset, alternative = "greater")
          print(res)

          # show sd
          print('sd')
          print(tapply(df_subset$min_mahalanobis_distance, df_subset$phase, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(min_mahalanobis_distance ~ phase, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'water speed:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'water speed:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}


[1] "-3"
[1] "R60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 R60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by phase
t = 4.1781, df = 34.634, p-value = 9.436e-05
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 0.4718952       Inf
sample estimates:
mean in group training_1 mean in group training_2 
                1.898224                 1.105786 

[1] "sd"
training_1 training_2 
 0.5314583  0.7024884 

Cohen's d

d estimate: 1.292842 (large)
95 percent confidence interval:
    lower     upper 
0.6283376 1.9573457 

[1] "L60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by phase
t = 3.933, df = 24.411, p-value = 0.0003044
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 0.

# STUDY B: Training Phase 1: early and late 2 trials

In [69]:

# Filter for t1 
df_single_t1 <- df_single[df_single$phase == 'training_1',]

speeds_list <- unique(df_single_t1$water_speed)
set_order_list <- unique(df_single_t1$set_order)
target_list <- unique(df_single_t1$target_id)


df_single_t1_agg <- aggregate(
  min_mahalanobis_distance ~ ppid_full + trial_set + water_speed + target_id,
  data = df_single_t1,
  FUN = mean
)



for (s in speeds_list) {
    print(s)

    p_vals <- c()
    
    for (t in target_list) {
        print(t)
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_single_t1_agg[df_single_t1_agg$water_speed == s & df_single_t1_agg$target_id == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(min_mahalanobis_distance ~ trial_set, data = df_subset)
          print(res)

          # show sd
          print('sd')
          print(tapply(df_subset$min_mahalanobis_distance, df_subset$trial_set, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(min_mahalanobis_distance ~ trial_set, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'water speed:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'water speed:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}


[1] "-3"
[1] "R60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 R60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by trial_set
t = 9.22, df = 33.144, p-value = 1.14e-10
alternative hypothesis: true difference in means between group early and group late is not equal to 0
95 percent confidence interval:
 1.412331 2.211947
sample estimates:
mean in group early  mean in group late 
          2.4448115           0.6326724 

[1] "sd"
    early      late 
0.7308609 0.4883051 

Cohen's d

d estimate: 2.915609 (large)
95 percent confidence interval:
   lower    upper 
1.996214 3.835005 

[1] "L60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by trial_set
t = 7.642, df = 20.834, p-value = 1.793e-07
alternative hypothesis: true difference in means between group early and group late is not equal to 0
95 percent confidence interval:
 0.8815176 1.5411043
sample estimates:
mean in gro

# STUDY B: Training Phase 2: early and late 2 trials

In [71]:

# Filter for t1 
df_single_t2 <- df_single[df_single$phase == 'training_2',]

speeds_list <- unique(df_single_t2$water_speed)
set_order_list <- unique(df_single_t2$set_order)
target_list <- unique(df_single_t2$target_id)


df_single_t2_agg <- aggregate(
  min_mahalanobis_distance ~ ppid_full + trial_set + water_speed + target_id,
  data = df_single_t2,
  FUN = mean
)



for (s in speeds_list) {
    print(s)

    p_vals <- c()
    
    for (t in target_list) {
        print(t)
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_single_t2_agg[df_single_t2_agg$water_speed == s & df_single_t2_agg$target_id == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(min_mahalanobis_distance ~ trial_set, data = df_subset)
          print(res)

          # show sd
          print('sd')
          print(tapply(df_subset$min_mahalanobis_distance, df_subset$trial_set, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(min_mahalanobis_distance ~ trial_set, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'water speed:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'water speed:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}


[1] "-3"
[1] "L60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by trial_set
t = 3.8718, df = 33.821, p-value = 0.0004698
alternative hypothesis: true difference in means between group early and group late is not equal to 0
95 percent confidence interval:
 0.1528136 0.4905985
sample estimates:
mean in group early  mean in group late 
          0.5730192           0.2513132 

[1] "sd"
    early      late 
0.3054628 0.2115940 

Cohen's d

d estimate: 1.224361 (large)
95 percent confidence interval:
    lower     upper 
0.5267871 1.9219355 

[1] "R60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 R60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by trial_set
t = 4.7928, df = 30, p-value = 4.18e-05
alternative hypothesis: true difference in means between group early and group late is not equal to 0
95 percent confidence interval:
 0.6325576 1.5718999
sample estimates:
mean i

# STUDY B: TRANSFER

In [73]:

# Filter for Transfer (Early trials only)
df_single_transfer <- df_single[df_single$trial_set == 'early',]
df_single_transfer$phase <- factor(df_single_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_single_transfer$water_speed)
target_list <- unique(df_single_transfer$target_id)

df_transfer_agg <- aggregate(
  min_mahalanobis_distance ~ ppid_full + phase + water_speed + target_id,
  data = df_single_transfer,
  FUN = mean
)



for (s in speeds_list) {
    print(s)

    p_vals <- c()
    
    for (t in target_list) {
        print(t)
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_transfer_agg[df_transfer_agg$water_speed == s & df_single_transfer$target_id == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(min_mahalanobis_distance ~ phase, data = df_subset, alternative = "greater")
          print(res)

          # show sd
          print('sd')
          print(tapply(df_subset$min_mahalanobis_distance, df_subset$phase, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(min_mahalanobis_distance ~ phase, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'water speed:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'water speed:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}


[1] "-3"
[1] "L60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by phase
t = 3.0913, df = 33.868, p-value = 0.001985
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 0.34219     Inf
sample estimates:
mean in group training_1 mean in group training_2 
                1.926703                 1.171239 

[1] "sd"
training_1 training_2 
 0.7098620  0.7557039 

Cohen's d

d estimate: 1.030448 (large)
95 percent confidence interval:
    lower     upper 
0.3094774 1.7514184 

[1] "R60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 R60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by phase
t = 2.8573, df = 33.928, p-value = 0.003624
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 0.36597